# LAB 10 — Fine-tuning and controlled experiments
AI-assisted reference notebook. The latest user request authorizes AI-authored work and a real full experiment. Numeric outputs and images come only from executing these cells; consult the saved run manifest and executed outputs for the actual status. Alignment, metric computation and automated error candidates are reference implementations under [29. AI Usage](README.md#29-ai-usage), not student-authored work. AI assistance remains disclosed; this is not represented as independently student-authored work.

Run this notebook from top to bottom after `tokenizer_alignment.ipynb` and `python -m unittest -v test_qa`. Each code cell states its input, operation and output. The stable cell IDs (e.g. `s14-run`) are used by the README. Other numbered sections are linked in the README.

## 9. Prediction Before Experiment
The AI-authored, pre-experiment [prediction.md](prediction.md) is frozen with the exact marker `PREDICTION_READY = YES`; full execution is explicitly enabled by the latest user authorization. The gate snapshots the prediction before downloading data or training and rejects subsequent edits. Never modify that prediction to match results; append comparisons in reflection instead.

In [1]:
# Input: workspace files and installed dependencies. Operation: configure local caches.
# Output: environment information on stdout; .runtime directories only.
from pathlib import Path
import os, sys, json, hashlib, random, platform, time, gc
from dataclasses import asdict, replace
from datetime import datetime, timezone
from importlib.metadata import version
ROOT = Path.cwd().resolve()
assert (ROOT / "lab_10_extractive_question_answering.md").is_file(), "Run with cwd=lab10"
RUNTIME = ROOT / ".runtime" / "full_run"
RUNTIME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(RUNTIME / "hf_cache")
os.environ["HF_DATASETS_CACHE"] = str(RUNTIME / "hf_cache" / "datasets")
os.environ["MPLCONFIGDIR"] = str(RUNTIME / "matplotlib")
sys.path.insert(0, str(ROOT))
import torch
from qa_model import Config, train_run, predict, baseline, squad_metrics
RUN_FULL = True  # Full experiment explicitly authorized in the latest user request.
BASE = Config(model_name="distilbert-base-uncased", max_length=384, stride=64,
              seed=42, batch_size=32, lr=3e-5, epochs=2, precision="bf16",
              runtime=str(RUNTIME), question_max_tokens=48, max_answer_length=30)
VERSIONS = {p: version(p) for p in ["torch", "transformers", "datasets", "numpy", "matplotlib", "tokenizers"]}
print({"python": sys.version, "versions": VERSIONS, "cuda_available": torch.cuda.is_available()})

{'python': '3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]', 'versions': {'torch': '2.14.0+cu132', 'transformers': '5.22.2', 'datasets': '5.1.0', 'numpy': '2.5.2', 'matplotlib': '3.11.2', 'tokenizers': '0.23.3'}, 'cuda_available': True}


In [2]:
# Input: RUN_FULL and pre-experiment prediction.md. Operation: freeze/check pre-experiment record.
# Output: .runtime/full_run/prediction_before_experiment.md and prediction_lock.json, never overwritten.
assert RUN_FULL, "Full training disabled. Explicit authorization and pre-experiment prediction are required."
prediction_bytes = (ROOT / "prediction.md").read_bytes()
assert "PREDICTION_READY = YES" in prediction_bytes.decode("utf-8").splitlines(), "Pre-experiment prediction not marked ready"
prediction_sha = hashlib.sha256(prediction_bytes).hexdigest()
lock_path = RUNTIME / "prediction_lock.json"
snapshot_path = RUNTIME / "prediction_before_experiment.md"
if lock_path.exists():
    lock = json.loads(lock_path.read_text(encoding="utf-8"))
    assert lock["sha256"] == prediction_sha, "Prediction changed after freeze; restore original, do not rewrite history"
    assert snapshot_path.exists() and hashlib.sha256(snapshot_path.read_bytes()).hexdigest() == prediction_sha
else:
    assert not snapshot_path.exists(), "Existing snapshot without lock: inspect manually before proceeding"
    snapshot_path.write_bytes(prediction_bytes)
    lock = {"sha256": prediction_sha, "frozen_at_utc": datetime.now(timezone.utc).isoformat()}
    lock_path.write_text(json.dumps(lock, indent=2), encoding="utf-8")
def assert_prediction_unchanged():
    assert hashlib.sha256((ROOT / "prediction.md").read_bytes()).hexdigest() == prediction_sha
print(lock)

{'sha256': 'e50b5d4c8b0dbf5bad9892b893610a34c0d6c163602c1601e4e878adcb8e2deb', 'frozen_at_utc': '2026-11-24T15:46:49.281774+00:00', 'authorship': 'AI-assisted under explicit user authorization', 'training_started': False}


## 3. Dataset
Use SQuAD 1.1 (`rajpurkar/squad`): question, context, annotated text and character start. Original training titles are shuffled with seed 42; 10% of titles become internal validation, 90% train. All questions from one title stay together, so overlapping windows and shared paragraphs cannot cross that boundary. The public SQuAD `validation` split is held-out evaluation/local test **not the official hidden test**. All three models finish training and internal-validation scoring before any held-out inference. Best epoch and best length are chosen only by internal validation F1 (EM then lower loss break ties; shorter length breaks remaining length ties). The entire held-out set is subsequently evaluated for the controlled comparison; the selected configuration drives Experiments 3–5 and perturbations. Do not tune against this evaluation set after viewing its scores.

The code asserts disjoint IDs, exact contexts and titles across all three splits. If an upstream revision violates this, execution stops rather than silently allowing leakage. Immutable dataset/model revisions, dataset fingerprints and full split IDs are recorded. No sample limit differs between length runs.

In [3]:
# Input: public SQuAD1 and DistilBERT repositories. Operation: pin revisions and split by title.
# Output: HF files in .runtime/hf_cache; .runtime/split_manifest.json and run_manifest.json.
from datasets import load_dataset
from huggingface_hub import HfApi, snapshot_download
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
api = HfApi()
DATASET_ID = "rajpurkar/squad"
MODEL_ID = BASE.model_name
previous_manifest_path = RUNTIME / "run_manifest.json"
previous_manifest = json.loads(previous_manifest_path.read_text(encoding="utf-8")) if previous_manifest_path.exists() else {}
bootstrap_path = RUNTIME / "bootstrap.json"
bootstrap = json.loads(bootstrap_path.read_text(encoding="utf-8")) if bootstrap_path.exists() else {}
DATASET_REVISION = previous_manifest.get("dataset_revision") or bootstrap.get("dataset_revision") or api.dataset_info(DATASET_ID).sha
MODEL_REVISION = previous_manifest.get("model_revision") or bootstrap.get("model_revision") or api.model_info(MODEL_ID).sha
raw = load_dataset(DATASET_ID, revision=DATASET_REVISION,
                   cache_dir=str(RUNTIME / "hf_cache" / "datasets"))
all_train = list(raw["train"])
eval_examples = list(raw["validation"])
titles = sorted({row["title"] for row in all_train})
random.Random(BASE.seed).shuffle(titles)
validation_titles = set(titles[:max(1, round(len(titles) * 0.1))])
train_examples = [row for row in all_train if row["title"] not in validation_titles]
validation_examples = [row for row in all_train if row["title"] in validation_titles]
splits = {"train": train_examples, "validation": validation_examples, "held_out_eval": eval_examples}
for key in ("id", "context", "title"):
    sets = [{row[key] for row in rows} for rows in splits.values()]
    assert all(not (sets[i] & sets[j]) for i in range(3) for j in range(i+1, 3)), f"Leakage: shared {key}"
for rows in splits.values():
    assert rows
    for row in rows:
        assert row["answers"]["text"], "SQuAD1 answerable core expected"
        assert all(row["context"][start:start+len(text)] == text
                   for text, start in zip(row["answers"]["text"], row["answers"]["answer_start"]))
manifest = {"dataset": DATASET_ID, "dataset_revision": DATASET_REVISION,
            "seed": BASE.seed, "group_key": "title", "validation_title_fraction": 0.1,
            "counts": {name: len(rows) for name, rows in splits.items()},
            "ids": {name: [row["id"] for row in rows] for name, rows in splits.items()},
            "fingerprints": {name: raw[name]._fingerprint for name in raw},
            "evaluation_note": "public SQuAD dev, local held-out evaluation; not official test"}
(RUNTIME / "split_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
model_source = bootstrap.get("model_source") or snapshot_download(MODEL_ID, revision=MODEL_REVISION,
    cache_dir=str(RUNTIME / "hf_cache" / "hub"),
    allow_patterns=["config.json", "tokenizer*", "vocab.txt", "special_tokens_map.json", "added_tokens.json", "model.safetensors", "model.safetensors.index.json", "model-*.safetensors"])
BASE = replace(BASE, model_name=model_source)
tokenizer = AutoTokenizer.from_pretrained(model_source, use_fast=True, local_files_only=True)
source_notebook = json.loads((ROOT / "qa_finetuning.ipynb").read_text(encoding="utf-8"))
notebook_code_hash = hashlib.sha256(json.dumps(["".join(c["source"]) for c in source_notebook["cells"] if c["cell_type"] == "code"], ensure_ascii=False).encode("utf-8")).hexdigest()
source_hashes = {name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest() for name in ("qa_model.py", "lab_10_extractive_question_answering.md")}
source_hashes["notebook_code"] = notebook_code_hash
run_manifest = {"started_at_utc": previous_manifest.get("started_at_utc") or datetime.now(timezone.utc).isoformat(), "python": sys.version,
    "versions": VERSIONS, "platform": platform.platform(), "cuda": torch.version.cuda,
    "hardware": torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor(),
    "model": MODEL_ID, "model_revision": MODEL_REVISION, "tokenizer": MODEL_ID,
    "dataset": DATASET_ID, "dataset_revision": DATASET_REVISION,
    "base_config": asdict(BASE), "optimizer": "torch.optim.AdamW", "prediction_sha256": prediction_sha,
    "split_manifest_sha256": hashlib.sha256((RUNTIME / "split_manifest.json").read_bytes()).hexdigest(),
    "source_hashes": source_hashes, "checkpoint_selection": "best internal validation F1, EM then lower loss ties; earliest epoch tie",
    "length_selection": "best internal validation F1, EM then lower loss, then shorter length; before held-out inference"}
run_identity = {key: run_manifest[key] for key in ("model_revision", "dataset_revision", "base_config", "prediction_sha256", "split_manifest_sha256", "source_hashes", "versions")}
run_id = hashlib.sha256(json.dumps(run_identity, sort_keys=True).encode("utf-8")).hexdigest()
if previous_manifest:
    assert previous_manifest.get("run_id") == run_id, "Existing full run identity differs; do not mix/rewrite checkpoints"
run_manifest["run_id"] = run_id
(RUNTIME / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
print(manifest["counts"], MODEL_REVISION, DATASET_REVISION)

Using the latest cached version of the dataset since rajpurkar/squad couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'plain_text' at C:\Users\tasdus_117\Desktop\NLP\lab10\.runtime\full_run\hf_cache\datasets\rajpurkar___squad\plain_text\0.0.0\7b6d24c440a36b6815f21b70d25016731768db1f (last modified on Tue Nov 24 22:48:54 2026).


{'train': 79214, 'validation': 8385, 'held_out_eval': 10570} 12040accade4e8a0f71eabdb258fecc2e7e948be 7b6d24c440a36b6815f21b70d25016731768db1f


## 14. Experiment 1: Context Length
Run three independent fine-tunings at `max_length=128/256/384` from the **same immutable initial model snapshot**. Fixed: original examples/splits/order, seed, question cap, tokenizer, overlap stride 64, batch size 32, learning rate 3e-5, AdamW settings, 2 epochs, max answer length 30, BF16 on CUDA (float32 CPU), validation-selected checkpoint rule. The 48-token question cap is applied equally: DistilBERT pair special tokens consume 3 slots, so even length 128 leaves at least 77 context tokens and stride 64 is valid (strictly smaller than context capacity). It also covers all spans of up to 30 tokens across neighboring windows. The feature builder validates capacity for the actual tokenizer.

Changing length necessarily changes the number of windows/optimizer steps per epoch; this is measured and reported, rather than falsely claiming equal compute. Larger context can add distractors and cost; the measured comparison is in results.csv and the completed error_analysis.md. Training time includes training epochs, validation inference/decoding and improved-epoch checkpoint writes; preprocessing, downloads, held-out inference and the final best-checkpoint copy are excluded, as recorded by `train_run`.

## 17. Experiment 2: Pretrained QA Model
`train_run` exposes the start/end logits and explicitly averages two cross-entropy losses. It logs epoch training and validation loss; QA evaluation uses original examples after merging every window. Internal validation EM/F1 are measured each epoch; the best epoch is saved by F1, EM then lower validation loss. All three lengths are selected before held-out inference. The backbone is pretrained; the QA head starts newly initialized, and each length run resets the same seed. See [qa_model.py](qa_model.py) for model/optimizer/loss and [tokenizer_alignment.ipynb](tokenizer_alignment.ipynb) for sections 10–16.

In [4]:
# Input: fixed train/internal-validation splits, immutable pretrained model and run identity.
# Operation: train or reuse three validated best-epoch checkpoints; select length WITHOUT held-out inference.
# Output: full_run/checkpoints/length_*/, length_runs.json and selection.json.
length_runs = []
for length in (128, 256, 384):
    assert_prediction_unchanged()
    config = replace(BASE, max_length=length)
    checkpoint = RUNTIME / "checkpoints" / f"length_{length}"
    provenance_path = checkpoint / "run_provenance.json"
    metadata_path = checkpoint / "training_metadata.json"
    provenance = {"run_id": run_id, "config": asdict(config)}
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        assert metadata.get("status") == "complete", f"Incomplete checkpoint requires inspection: {checkpoint}"
        assert provenance_path.exists() and json.loads(provenance_path.read_text(encoding="utf-8")) == provenance
        assert metadata["config"] == asdict(config), "Checkpoint configuration mismatch"
        assert (checkpoint / "config.json").exists() and (checkpoint / "model.safetensors").exists()
        print("Reusing complete checkpoint", length, metadata["best_epoch"])
    else:
        assert not checkpoint.exists() or not any(checkpoint.iterdir()), f"Nonempty checkpoint has no complete metadata: {checkpoint}"
        run = train_run(train_examples, validation_examples, config, checkpoint)
        metadata = {key:value for key,value in run.items() if key not in ("model", "tokenizer")}
        assert metadata["status"] == "complete"
        provenance_path.write_text(json.dumps(provenance, indent=2), encoding="utf-8")
        del run
    (checkpoint / "experiment_config.json").write_text(json.dumps(asdict(config), indent=2), encoding="utf-8")
    length_runs.append({"model": f"Transformer_{length}", "max_length": length,
        "history": metadata["history"], "best_epoch": metadata["best_epoch"],
        "best_validation": metadata["best_validation"], "training_time": metadata["training_time"],
        "checkpoint": str(checkpoint.relative_to(ROOT)), "config": asdict(config), "run_id": run_id})
    (RUNTIME / "length_runs.json").write_text(json.dumps(length_runs, indent=2), encoding="utf-8")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
selected_run = max(length_runs, key=lambda r: (r["best_validation"]["f1"], r["best_validation"]["em"],
                                               -r["best_validation"]["loss"], -r["max_length"]))
selected_length = selected_run["max_length"]
selected_name = selected_run["model"]
selected_config = Config(**selected_run["config"])
selection = {"run_id": run_id, "selected_length": selected_length, "selected_model": selected_name,
    "best_epoch": selected_run["best_epoch"], "validation_metrics": selected_run["best_validation"],
    "selected_at_utc": datetime.now(timezone.utc).isoformat(), "held_out_inference_started": False}
(RUNTIME / "selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
run_manifest["main_analysis_length"] = selected_length
run_manifest["selected_checkpoint"] = selected_run["checkpoint"]
(RUNTIME / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
assert_prediction_unchanged()
print("Selected before held-out inference:", selection)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: C:\Users\tasdus_117\Desktop\NLP\lab10\.runtime\full_run\hf_cache\hub\models--distilbert-base-uncased\snapshots\12040accade4e8a0f71eabdb258fecc2e7e948be
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Preparing length=128: train=79214, validation=8385


Features train=192549, validation=20352, steps/epoch=6018, precision=bf16_autocast


C:\Users\tasdus_117\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\autograd\graph.py:1072: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\transformers\cuda\attention_backward.cu:912.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


length=128 epoch=1/2 step=100/6018 loss=1.70364 elapsed=21.5s


... (252 outputs left out)

{'epoch': 2, 'train_loss': 0.9854316201885757, 'validation_loss': 0.9780842387845731, 'validation_em': 0.6621347644603458, 'validation_f1': 0.7995998961844997, 'optimizer_steps': 5006, 'epoch_seconds': 441.97083700000076}


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Selected before held-out inference: {'run_id': '8faacbbd16e7d738dcf71bbdd8f51d12ffe7b805c1b43f8ea62df17775b0772c', 'selected_length': 384, 'selected_model': 'Transformer_384', 'best_epoch': 2, 'validation_metrics': {'em': 0.6621347644603458, 'f1': 0.7995998961844997, 'loss': 0.9780842387845731}, 'selected_at_utc': '2026-11-24T16:31:46.254178+00:00', 'held_out_inference_started': False}


In [5]:
# Input: all complete best-epoch checkpoints and already frozen validation selection.
# Operation: evaluate all three lengths on public held-out dev; no tuning or reselection.
# Output: full_run/predictions_length_*.json and updated selection.json.
all_predictions = {}
selection["held_out_inference_started"] = True
(RUNTIME / "selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
for run in length_runs:
    checkpoint = ROOT / run["checkpoint"]
    config = Config(**run["config"])
    model = AutoModelForQuestionAnswering.from_pretrained(checkpoint, local_files_only=True)
    run_tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True, local_files_only=True)
    predictions = predict(model, run_tokenizer, eval_examples, config,
                          device="cuda" if torch.cuda.is_available() else "cpu")
    assert set(predictions) == {row["id"] for row in eval_examples}
    all_predictions[run["model"]] = predictions
    (RUNTIME / f"predictions_length_{run['max_length']}.json").write_text(
        json.dumps({"run_id":run_id,"predictions":predictions}, ensure_ascii=False, indent=2), encoding="utf-8")
    del model, run_tokenizer, predictions
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("Held-out inference complete; selected configuration remains", selected_length)


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Held-out inference complete; selected configuration remains 384


## 18. Exact Match
The reference metric lowercases, strips ASCII punctuation, removes English articles and collapses whitespace, matching the SQuAD evaluation convention. Multiple annotated answers are accepted: EM and F1 each take their maximum across references. Aggregation is a macro average per original question, never per window. `squad_metrics` returns fractions; figures and `results.csv` show percentages.

## 19. Token-level F1 for QA
Token overlap uses **multiset counts**, so duplicate tokens cannot be counted more often than they occur in either answer. This metric uses normalized whitespace tokens, not Transformer subwords. Completed hand-calculated examples, including repeated tokens and multiple golds, are in [calculations.md](calculations.md).

## 21. Baseline
The heuristic in `qa_model.baseline` chooses a candidate sentence/span using lexical overlap. It receives only question/context, never gold answers. It is evaluated on the exact same held-out examples. It has no train loss or training time; empty CSV loss/time fields mean not applicable, not zero measured loss.

In [6]:
# Input: baseline and all Transformer predictions. Operation: evaluate against every gold reference.
# Output: root results.csv, .runtime/full_run/predictions.json; stdout measured summary.
import csv, statistics, re
all_predictions["Baseline"] = {row["id"]: baseline({"id": row["id"], "question": row["question"], "context": row["context"]}) for row in eval_examples}
def content_words(text):
    stop = {"a", "an", "the", "is", "are", "was", "were", "of", "in", "on", "at", "to", "for", "and", "or", "what", "which", "who", "when", "where", "how", "why", "does", "did", "do"}
    return set(re.findall(r"[a-z0-9]+", text.lower())) - stop
def lexical_recall(question, text):
    words = content_words(question)
    return len(words & content_words(text)) / len(words) if words else 0.0
def length_bin(n):
    return "1" if n <= 1 else "2-3" if n <= 3 else "4-6" if n <= 6 else "7+"
evaluation_rows = []
for model_name, predictions in all_predictions.items():
    for example in eval_examples:
        pred = predictions[example["id"]]
        golds = example["answers"]["text"]
        offsets = list(zip(example["answers"]["answer_start"],
                           [a+len(t) for a,t in zip(example["answers"]["answer_start"], golds)]))
        # Choose one accepted gold span jointly; never combine endpoints from different references.
        gold_index = min(range(len(offsets)), key=lambda k: (
            int(pred["start"] != offsets[k][0]) + int(pred["end"] != offsets[k][1]),
            abs(pred["start"] - offsets[k][0]) + abs(pred["end"] - offsets[k][1]), k))
        gold_start, gold_end = offsets[gold_index]
        wrong_start = pred["start"] != gold_start
        wrong_end = pred["end"] != gold_end
        metrics = squad_metrics(pred["text"], golds)
        assert 0 <= metrics["em"] <= 1 and 0 <= metrics["f1"] <= 1
        assert example["context"][pred["start"]:pred["end"]] == pred["text"]
        answer_tokens = len(tokenizer(golds[0], add_special_tokens=False)["input_ids"])
        overlap_context = lexical_recall(example["question"], example["context"])
        overlap_answer = max(lexical_recall(example["question"], text) for text in golds)
        evaluation_rows.append({"run_id": run_id, "model": model_name, "id": example["id"], "question": example["question"],
            "context": example["context"], "gold": golds, "gold_offsets": offsets, "prediction": pred,
            **metrics, "comparison_gold_index": gold_index, "comparison_gold_offsets": offsets[gold_index],
            "wrong_start": wrong_start, "wrong_end": wrong_end,
            "wrong_span": wrong_start and wrong_end,
            "span_mismatch": (pred["start"],pred["end"]) not in offsets,
            "answer_length_tokens": answer_tokens, "answer_length_bin": length_bin(answer_tokens),
            "context_overlap": overlap_context, "answer_overlap": overlap_answer,
            "context_overlap_group": "high" if overlap_context >= 0.5 else "low",
            "answer_overlap_group": "high" if overlap_answer >= 0.5 else "low"})
def aggregate(rows):
    return {"n": len(rows), "EM": 100*statistics.mean(r["em"] for r in rows) if rows else None,
            "F1": 100*statistics.mean(r["f1"] for r in rows) if rows else None}
summary = []
for name in ["Baseline", "Transformer_128", "Transformer_256", "Transformer_384"]:
    row = {"model": name, **aggregate([r for r in evaluation_rows if r["model"] == name])}
    if name == "Baseline":
        row.update(max_length="", train_loss="", validation_loss="", training_time="", best_epoch="", validation_EM="", validation_F1="", selected=False)
    else:
        run = next(r for r in length_runs if r["model"] == name)
        best_history = next(h for h in run["history"] if h["epoch"] == run["best_epoch"])
        row.update(max_length=run["max_length"], train_loss=best_history["train_loss"],
                   validation_loss=best_history["validation_loss"], training_time=run["training_time"],
                   best_epoch=run["best_epoch"], validation_EM=100*run["best_validation"]["em"],
                   validation_F1=100*run["best_validation"]["f1"], selected=run["max_length"] == selected_length)
    summary.append(row)
with (ROOT / "results.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["model", "max_length", "n", "EM", "F1", "train_loss", "validation_loss", "training_time", "best_epoch", "validation_EM", "validation_F1", "selected"])
    writer.writeheader()
    writer.writerows(summary)
(RUNTIME / "predictions.json").write_text(json.dumps(evaluation_rows, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

[
  {
    "model": "Baseline",
    "n": 10570,
    "EM": 0.21759697256385999,
    "F1": 18.883403238292658,
    "max_length": "",
    "train_loss": "",
    "validation_loss": "",
    "training_time": "",
    "best_epoch": "",
    "validation_EM": "",
    "validation_F1": "",
    "selected": false
  },
  {
    "model": "Transformer_128",
    "n": 10570,
    "EM": 75.08041627246925,
    "F1": 83.15396477033919,
    "max_length": 128,
    "train_loss": 0.6645146589969695,
    "validation_loss": 0.8422786516588439,
    "training_time": 763.5948074000007,
    "best_epoch": 2,
    "validation_EM": 65.39057841383423,
    "validation_F1": 78.60057960994502,
    "selected": false
  },
  {
    "model": "Transformer_256",
    "n": 10570,
    "EM": 75.61967833491012,
    "F1": 84.09703031926902,
    "max_length": 256,
    "train_loss": 0.9443520413784287,
    "validation_loss": 0.9586139172854489,
    "training_time": 627.8613790999989,
    "best_epoch": 2,
    "validation_EM": 65.61717352415026,


## 20. Experiment 3: EM vs F1
The primary comparison is Baseline versus the internal-validation-selected fine-tuned Transformer; the figure also shows the two other controlled length runs. The following cell extracts actual `EM=0, F1>0` examples and flags whether at least five were observed. No result is fabricated to fill a quota. Read the original context and write your own explanation of boundary/partial overlap in [error_analysis.md](error_analysis.md).

In [7]:
# Input: measured per-example scores. Operation: select partial-match cases, draw real curves/metrics.
# Output: .runtime/full_run/partial_match_candidates.json, learning_curves.png, em_f1_analysis.png.
import matplotlib.pyplot as plt
main_rows = [r for r in evaluation_rows if r["model"] == selected_name]
partial = [r for r in main_rows if r["em"] == 0 and r["f1"] > 0]
partial_report = {"observed_count": len(partial), "required_minimum": 5,
    "sufficient": len(partial) >= 5, "examples": partial[:10], "explanations": "AI-authored evidence-grounded explanations are recorded in error_analysis.md"}
(RUNTIME / "partial_match_candidates.json").write_text(json.dumps(partial_report, ensure_ascii=False, indent=2), encoding="utf-8")
print({k:v for k,v in partial_report.items() if k != "examples"})
for row in partial[:5]:
    print(row["question"], row["gold"], row["prediction"]["text"], row["f1"])
fig, axes = plt.subplots(2, 3, figsize=(13, 7), constrained_layout=True)
for column, run in enumerate(length_runs):
    ax = axes[0, column]
    history = run["history"]
    ax.plot([h["epoch"] for h in history], [h["train_loss"] for h in history], marker="o", label="train")
    ax.plot([h["epoch"] for h in history], [h["validation_loss"] for h in history], marker="o", label="validation")
    ax.set(title=f"max_length={run['max_length']}", xlabel="Epoch", ylabel="Mean start/end loss")
    ax.axvline(run["best_epoch"], color="gray", linestyle=":", label="selected epoch")
    ax.legend()
    score_ax = axes[1, column]
    score_ax.plot([h["epoch"] for h in history], [100*h["validation_em"] for h in history], marker="o", label="validation EM")
    score_ax.plot([h["epoch"] for h in history], [100*h["validation_f1"] for h in history], marker="o", label="validation F1")
    score_ax.set(xlabel="Epoch", ylabel="Internal validation score (%)", ylim=(0,100))
    score_ax.axvline(run["best_epoch"], color="gray", linestyle=":")
    score_ax.legend()
fig.savefig(ROOT / "learning_curves.png", dpi=160)
plt.close(fig)
fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
x = list(range(len(summary)))
ax.bar([i-.18 for i in x], [r["EM"] for r in summary], width=.36, label="EM")
ax.bar([i+.18 for i in x], [r["F1"] for r in summary], width=.36, label="F1")
ax.set(xticks=x, xticklabels=[r["model"] for r in summary], ylabel="Score (%)", ylim=(0,100), title="Held-out public dev (not official test)")
ax.legend()
fig.savefig(ROOT / "em_f1_analysis.png", dpi=160)
plt.close(fig)

{'observed_count': 1507, 'required_minimum': 5, 'sufficient': True, 'explanations': 'AI-authored evidence-grounded explanations are recorded in error_analysis.md'}
What city did Super Bowl 50 take place in? ['Santa Clara', 'Santa Clara', 'Santa Clara'] Santa Clara, California 0.8
What city did Super Bowl 50 take place in? ['Santa Clara', 'Santa Clara', 'Santa Clara'] Santa Clara, California 0.8
The name of the NFL championship game is? ['Super Bowl', 'Super Bowl', 'Super Bowl'] Super Bowl L 0.8
How many teams have been in the Super Bowl eight times? ['four', 'four', 'four'] one of four 0.5
Which Newton turnover resulted in seven points for Denver? ['a fumble', 'a fumble', 'fumble'] fumble which they recovered for a touchdown 0.2857142857142857


## 22. Experiment 4: Question/Context Matching
Operational lexical overlap is content-word recall: intersection of lowercase alphanumeric word sets divided by question content words (listed stop words removed). High is >=0.5, low is <0.5; thresholds are fixed before viewing scores. Two views are exported: question–full-context overlap and question–gold-answer wording overlap, plus their four intersections. For each nonempty group the notebook compares Baseline and the validation-selected Transformer EM/F1 and shows real examples. Empty groups remain `null` with `n=0`.

This stratification measures correlation, not causal dependence, and low lexical overlap does not prove semantic paraphrase. Full-context overlap can be high because of distractors; inspect each example and annotate lexical matching versus semantic matching independently. There are no generated paraphrases or unsupported error explanations.

In [8]:
# Input: held-out scores and fixed lexical groups. Operation: aggregate both models by overlap.
# Output: .runtime/full_run/lexical_overlap.json and stdout actual examples for independent inspection.
lexical_groups = []
for model_name in ("Baseline", selected_name):
    model_rows = [r for r in evaluation_rows if r["model"] == model_name]
    for field in ("context_overlap_group", "answer_overlap_group"):
        for group in ("low", "high"):
            selected = [r for r in model_rows if r[field] == group]
            lexical_groups.append({"model": model_name, "view": field, "group": group,
                **aggregate(selected), "example_ids": [r["id"] for r in selected[:5]]})
    for context_group in ("low", "high"):
        for answer_group in ("low", "high"):
            selected = [r for r in model_rows if r["context_overlap_group"] == context_group and r["answer_overlap_group"] == answer_group]
            lexical_groups.append({"model": model_name, "view": "context_x_answer",
                "group": f"{context_group}/{answer_group}", **aggregate(selected),
                "example_ids": [r["id"] for r in selected[:5]]})
(RUNTIME / "lexical_overlap.json").write_text(json.dumps(lexical_groups, indent=2), encoding="utf-8")
print(json.dumps(lexical_groups, indent=2))
for group in ("low", "high"):
    print("Question-context", group)
    for row in [r for r in main_rows if r["context_overlap_group"] == group][:3]:
        print(row["id"], row["question"], row["gold"], row["prediction"]["text"])

[
  {
    "model": "Baseline",
    "view": "context_overlap_group",
    "group": "low",
    "n": 1605,
    "EM": 0.12461059190031153,
    "F1": 15.723138587293528,
    "example_ids": [
      "56bf10f43aeaaa14008c9501",
      "56beaa4a3aeaaa14008c91c2",
      "56bf159b3aeaaa14008c9507",
      "56bf159b3aeaaa14008c9508",
      "56bf159b3aeaaa14008c9509"
    ]
  },
  {
    "model": "Baseline",
    "view": "context_overlap_group",
    "group": "high",
    "n": 8965,
    "EM": 0.2342442833240379,
    "F1": 19.449184026341023,
    "example_ids": [
      "56be4db0acb8001400a502ec",
      "56be4db0acb8001400a502ed",
      "56be4db0acb8001400a502ee",
      "56be4db0acb8001400a502ef",
      "56be4db0acb8001400a502f0"
    ]
  },
  {
    "model": "Baseline",
    "view": "answer_overlap_group",
    "group": "low",
    "n": 10404,
    "EM": 0.03844675124951942,
    "F1": 18.324020278786524,
    "example_ids": [
      "56be4db0acb8001400a502ec",
      "56be4db0acb8001400a502ed",
      "56be4db0acb800

### Prediction 3/4 checks: supplied-context noise and sentence-order perturbation
Input is every held-out example whose **all** annotated gold spans lie wholly inside a single regex sentence segment. Sentence cuts follow punctuation plus whitespace; abbreviations can be split, so this is an operational order perturbation, not a linguistic sentence parser. Each original segment is retained verbatim; reverse segment order and insert newlines between segments, then remap every gold offset and assert all gold texts are unchanged. Cases with any gold crossing a cut are excluded and reported, never silently assigned a corrupt label.

For this common eligible subset, compare original context, a fixed synthetic irrelevant prefix, and reordered context using the same validation-selected checkpoint/configuration. The prefix is a controlled perturbation, not a fabricated observed dataset result. No training/tuning uses these variants. All eligible held-out examples are included with no arbitrary cap. EM/F1 and paired score changes are measured against each variant's updated annotations. Inspect these outputs to compare the unchanged predictions; a change does not itself prove a particular cause.


In [9]:
# Input: every held-out example, selected trained checkpoint, fixed perturbation rules.
# Operation: preserve gold substrings/offsets, infer three paired variants and compare actual EM/F1.
# Output: full_run/perturbation_results.json with summaries, paired rows and explicit exclusions.
NOISE_PREFIX = ("The purple notebook rests beside a quiet window. "
    "An imaginary clock marks an ordinary afternoon. "
    "A spare pencil lies next to a closed drawer. "
    "These introductory sentences are unrelated background material.\n")
def reverse_context(example):
    context = example["context"]
    cuts = [0] + [match.end() for match in re.finditer(r"(?<=[.!?])\s+", context)] + [len(context)]
    segments = [(a,b) for a,b in zip(cuts,cuts[1:]) if b>a]
    owners = []
    for start,text in zip(example["answers"]["answer_start"], example["answers"]["text"]):
        owner = next((i for i,(a,b) in enumerate(segments) if a <= start and start+len(text) <= b), None)
        if owner is None:
            return None, "at least one accepted gold crosses a sentence cut"
        owners.append(owner)
    new_parts, new_segment_starts = [], {}
    cursor = 0
    for i in reversed(range(len(segments))):
        a,b = segments[i]
        new_segment_starts[i] = cursor
        new_parts.append(context[a:b])
        cursor += b-a+1  # one inserted newline between verbatim segments
    new_context = "\n".join(new_parts)
    new_starts = [new_segment_starts[owner] + start-segments[owner][0]
        for owner,start in zip(owners,example["answers"]["answer_start"])]
    result = {**example, "context":new_context,
        "answers":{"text":list(example["answers"]["text"]),"answer_start":new_starts}}
    assert all(new_context[a:a+len(t)] == t for a,t in zip(new_starts,result["answers"]["text"]))
    return result, None
original_variant, noise_variant, order_variant, exclusions = [], [], [], []
for example in eval_examples:
    reordered, reason = reverse_context(example)
    if reordered is None:
        exclusions.append({"id":example["id"],"reason":reason})
        continue
    original_variant.append(example)
    noisy = {**example, "context":NOISE_PREFIX + example["context"],
        "answers":{"text":list(example["answers"]["text"]),
                   "answer_start":[a+len(NOISE_PREFIX) for a in example["answers"]["answer_start"]]}}
    assert all(noisy["context"][a:a+len(t)] == t for a,t in zip(noisy["answers"]["answer_start"],noisy["answers"]["text"]))
    noise_variant.append(noisy)
    order_variant.append(reordered)
assert original_variant, "No eligible paired examples"
selected_checkpoint = ROOT / selected_run["checkpoint"]
selected_model = AutoModelForQuestionAnswering.from_pretrained(selected_checkpoint, local_files_only=True)
selected_tokenizer = AutoTokenizer.from_pretrained(selected_checkpoint, use_fast=True, local_files_only=True)
variants = {"original":original_variant,"noise_prefix":noise_variant,"sentence_reverse":order_variant}
variant_predictions = {"original":{row["id"]:all_predictions[selected_name][row["id"]] for row in original_variant}}
for variant_name in ("noise_prefix","sentence_reverse"):
    variant_predictions[variant_name] = predict(selected_model, selected_tokenizer, variants[variant_name],
        selected_config, device="cuda" if torch.cuda.is_available() else "cpu")
paired_rows, perturbation_summary = [], {}
for variant_name, examples in variants.items():
    rows = []
    for example in examples:
        prediction = variant_predictions[variant_name][example["id"]]
        metric = squad_metrics(prediction["text"], example["answers"]["text"])
        original_score = squad_metrics(all_predictions[selected_name][example["id"]]["text"], example["answers"]["text"])
        assert example["context"][prediction["start"]:prediction["end"]] == prediction["text"]
        row = {"run_id":run_id,"variant":variant_name,"id":example["id"],"question":example["question"],
            "context":example["context"],"gold":example["answers"],"prediction":prediction,**metric,
            "delta_em":metric["em"]-original_score["em"],"delta_f1":metric["f1"]-original_score["f1"],
            "prediction_text_changed":prediction["text"] != all_predictions[selected_name][example["id"]]["text"]}
        rows.append(row)
    paired_rows.extend(rows)
    perturbation_summary[variant_name] = {**aggregate(rows),
        "mean_delta_EM":100*statistics.mean(r["delta_em"] for r in rows),
        "mean_delta_F1":100*statistics.mean(r["delta_f1"] for r in rows),
        "F1_improved":sum(r["delta_f1"]>0 for r in rows),"F1_worsened":sum(r["delta_f1"]<0 for r in rows),
        "prediction_text_changed":sum(r["prediction_text_changed"] for r in rows)}
perturbation_report = {"run_id":run_id,"selected_model":selected_name,"selected_length":selected_length,
    "eligible_count":len(original_variant),"total_heldout":len(eval_examples),"excluded_count":len(exclusions),
    "exclusions":exclusions,"noise_prefix":NOISE_PREFIX,"sentence_cut_regex":r"(?<=[.!?])\s+",
    "summary":perturbation_summary,"rows":paired_rows}
(RUNTIME / "perturbation_results.json").write_text(json.dumps(perturbation_report,ensure_ascii=False,indent=2),encoding="utf-8")
del selected_model,selected_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(json.dumps({k:v for k,v in perturbation_report.items() if k not in ("rows","exclusions")},indent=2))


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

{
  "run_id": "8faacbbd16e7d738dcf71bbdd8f51d12ffe7b805c1b43f8ea62df17775b0772c",
  "selected_model": "Transformer_384",
  "selected_length": 384,
  "eligible_count": 10438,
  "total_heldout": 10570,
  "excluded_count": 132,
  "noise_prefix": "The purple notebook rests beside a quiet window. An imaginary clock marks an ordinary afternoon. A spare pencil lies next to a closed drawer. These introductory sentences are unrelated background material.\n",
  "sentence_cut_regex": "(?<=[.!?])\\s+",
  "summary": {
    "original": {
      "n": 10438,
      "EM": 76.56639202912436,
      "F1": 84.69980080226495,
      "mean_delta_EM": 0.0,
      "mean_delta_F1": 0.0,
      "F1_improved": 0,
      "F1_worsened": 0,
      "prediction_text_changed": 0
    },
    "noise_prefix": {
      "n": 10438,
      "EM": 75.54129143514083,
      "F1": 83.77437998416984,
      "mean_delta_EM": -1.0251005939835216,
      "mean_delta_F1": -0.9254208180950978,
      "F1_improved": 594,
      "F1_worsened": 738,
   

## 23. Experiment 5: Answer Length
Groups are `1`, `2-3`, `4-6`, `7+` **Transformer subword tokens** in the first annotated gold answer, without special tokens, with the same tokenizer for every model. This differs from normalized metric words; the convention is explicit and fixed. All gold alternatives are still used for EM/F1. Empty groups are displayed as gaps (`NaN`), never fabricated zero scores. The decoder cap is 30 subwords, so longer gold answers may be impossible to recover exactly; inspect that limitation when interpreting the 7+ group. Compare observed results with the unchanged prediction in [prediction.md](prediction.md).

In [10]:
# Input: actual evaluation rows and fixed gold-token length bins. Operation: aggregate/plot scores.
# Output: .runtime/full_run/answer_length.json and root answer_length_analysis.png.
bins = ("1", "2-3", "4-6", "7+")
length_groups = []
for name in ("Baseline", selected_name):
    for group in bins:
        selected = [r for r in evaluation_rows if r["model"] == name and r["answer_length_bin"] == group]
        length_groups.append({"model": name, "answer_length": group, **aggregate(selected)})
(RUNTIME / "answer_length.json").write_text(json.dumps(length_groups, indent=2), encoding="utf-8")
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for ax, metric in zip(axes, ("EM", "F1")):
    for name in ("Baseline", selected_name):
        rows = [r for r in length_groups if r["model"] == name]
        ax.plot(bins, [r[metric] if r[metric] is not None else float("nan") for r in rows], marker="o", label=name)
    ax.set(xlabel="Gold answer length (subwords)", ylabel=f"{metric} (%)", ylim=(0,100))
    ax.legend()
fig.savefig(ROOT / "answer_length_analysis.png", dpi=160)
plt.close(fig)
print(json.dumps(length_groups, indent=2))
print("Gold answers longer than decoding cap:", sum(r["answer_length_tokens"] > BASE.max_answer_length for r in main_rows))

[
  {
    "model": "Baseline",
    "answer_length": "1",
    "n": 2186,
    "EM": 0.0,
    "F1": 10.469838905297557
  },
  {
    "model": "Baseline",
    "answer_length": "2-3",
    "n": 3988,
    "EM": 0.10030090270812438,
    "F1": 15.5620128109366
  },
  {
    "model": "Baseline",
    "answer_length": "4-6",
    "n": 2840,
    "EM": 0.07042253521126761,
    "F1": 20.910825161837106
  },
  {
    "model": "Baseline",
    "answer_length": "7+",
    "n": 1556,
    "EM": 1.0925449871465296,
    "F1": 35.51571583042441
  },
  {
    "model": "Transformer_384",
    "answer_length": "1",
    "n": 2186,
    "EM": 79.73467520585544,
    "F1": 84.85627877180212
  },
  {
    "model": "Transformer_384",
    "answer_length": "2-3",
    "n": 3988,
    "EM": 80.09027081243731,
    "F1": 86.32217877408775
  },
  {
    "model": "Transformer_384",
    "answer_length": "4-6",
    "n": 2840,
    "EM": 77.04225352112675,
    "F1": 85.84442554861997
  },
  {
    "model": "Transformer_384",
    "answer_leng

## 24. Error Analysis
The next cell extracts real candidates: at least five wrong starts, five wrong ends and five wrong spans **if those counts exist**. For each example choose one annotated gold pair jointly: fewest mismatched endpoints, then minimum total absolute character distance, then annotation order. Start/end errors are measured against that same pair and can overlap. A repeated answer at an unannotated location can receive EM=1 yet be an offset error, so inspect such cases rather than calling them semantic failures automatically. Wrong span means both boundaries are wrong relative to the jointly selected gold pair. The separate `span_mismatch` flag means the predicted character pair matches no annotated pair. Boundaries from different references are never combined into an invented gold span.

The machine only supplies candidates and counts. AI will read actual context and add explicitly AI-authored an evidence-backed type/explanation in [error_analysis.md](error_analysis.md): wrong start/end/boundary, partial answer, distractor, insufficient context, long context, multiple plausible spans, question ambiguity, truncation, tokenization. A missing context is not assumed merely because a prediction is wrong. Check feature coverage before claiming truncation.

## 25. Wrong Start vs Wrong End
Report counts separately and develop your own hypothesis after inspecting examples. A wrong start with correct end differs from a correct start with wrong end; both-wrong is another group. The exported count matrix preserves these distinctions. It does not supply speculative causes.

In [11]:
# Input: actual validation-selected model predictions and all gold character offsets. Operation: select/count errors.
# Output: .runtime/full_run/error_candidates.json and error_counts.json; no analysis text overwritten.
error_candidates = {}
for flag in ("wrong_start", "wrong_end", "wrong_span"):
    selected = [r for r in main_rows if r[flag]]
    error_candidates[flag] = {"observed_count": len(selected), "required_minimum": 5,
        "sufficient": len(selected) >= 5, "examples": selected[:10]}
error_counts = {flag: sum(r[flag] for r in main_rows) for flag in error_candidates}
error_counts["span_mismatch"] = sum(r["span_mismatch"] for r in main_rows)
error_counts["n"] = len(main_rows)
error_counts["boundary_matrix"] = {f"start_{'wrong' if start_wrong else 'correct'}_end_{'wrong' if end_wrong else 'correct'}":
    sum(r["wrong_start"] == start_wrong and r["wrong_end"] == end_wrong for r in main_rows)
    for start_wrong in (False,True) for end_wrong in (False,True)}
(RUNTIME / "error_candidates.json").write_text(json.dumps(error_candidates, ensure_ascii=False, indent=2), encoding="utf-8")
(RUNTIME / "error_counts.json").write_text(json.dumps(error_counts, indent=2), encoding="utf-8")
print(json.dumps(error_counts, indent=2))
print({name: {k:v for k,v in report.items() if k != "examples"} for name,report in error_candidates.items()})

{
  "wrong_start": 2245,
  "wrong_end": 1924,
  "wrong_span": 1302,
  "span_mismatch": 2867,
  "n": 10570,
  "boundary_matrix": {
    "start_correct_end_correct": 7703,
    "start_correct_end_wrong": 622,
    "start_wrong_end_correct": 943,
    "start_wrong_end_wrong": 1302
  }
}
{'wrong_start': {'observed_count': 2245, 'required_minimum': 5, 'sufficient': True}, 'wrong_end': {'observed_count': 1924, 'required_minimum': 5, 'sufficient': True}, 'wrong_span': {'observed_count': 1302, 'required_minimum': 5, 'sufficient': True}}


## 26. No-Answer Extension
SQuAD 2.0 is optional and not implemented in this core run. A feature whose gold falls outside its window is trained to CLS; that sentinel does **not** implement calibrated document-level unanswerability. Core inference forces a valid context span. Never interpret its raw logit sum as probability that an answer exists.

## 27. Application: Document QA
After validation selection and training, run in a terminal:
```powershell
python document_qa.py --checkpoint .runtime/full_run/checkpoints/length_384 --question "What is the capital of France?" --context "The capital of France is Paris."
python document_qa.py --checkpoint .runtime/full_run/checkpoints/length_384 --question "Your question" --document path/to/document.txt
```
Input is a supplied question and UTF-8 text/document; output is JSON to stdout with extracted text and `[start,end)` character offsets. Long documents use the same window merge; no retrieval or RAG is added.

## 29. AI Usage
AI-generated: training boilerplate, cache/split orchestration, visualization, direct alignment/metric/decoding reference code and automated error candidates. The latest user request authorizes AI to complete calculations, prediction, error analysis, reflection and learning checks. All remain honestly labeled AI-authored; observed results must cite the actual run artifacts. No student personal experience or unobserved empirical conclusion is asserted.

## 30. Reproducibility
Model/tokenizer: immutable DistilBERT snapshot recorded in run manifest. Dataset: immutable SQuAD1 revision, split IDs/fingerprints recorded. Train/validation: seeded title split of original train; local held-out evaluation: public dev, never official test. Lengths 128/256/384; stride64; question cap48; batch32; learning rate3e-5; epochs2; AdamW; seed42; exact loss, optimizer settings, hardware/device, actual feature counts and timing are recorded by `train_run` in each checkpoint's `training_metadata.json`. `experiment_config.json` drives the CLI. `.runtime/full_run/hf_cache` stores downloaded data/weights; `.runtime/full_run/checkpoints/length_*` stores trained weights. Reuse saved revisions/IDs for exact replay, rather than assuming upstream `main` is unchanged. No bitwise cross-hardware determinism is promised.

## 31. Deliverables
The completed full run produced `results.csv`, `learning_curves.png`, `em_f1_analysis.png`, `answer_length_analysis.png`. The notebook also saves actual predictions, candidate counts, overlap/length summaries and reproducibility metadata in `.runtime`. The saved outputs are actual computations, not placeholders. See README for all 36 mapped sections.

In [12]:
# Input: full run outputs and original prediction. Operation: verify expected artifacts/provenance.
# Output: updated .runtime/run_manifest.json and artifact file sizes on stdout.
assert_prediction_unchanged()
required = ["results.csv", "learning_curves.png", "em_f1_analysis.png", "answer_length_analysis.png"]
assert all((ROOT / name).is_file() and (ROOT / name).stat().st_size > 0 for name in required)
run_manifest["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
run_manifest["training_times_seconds"] = {r["model"]: r["training_time"] for r in length_runs}
run_manifest["artifacts_sha256"] = {name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest() for name in required}
run_manifest["analysis_status"] = "AI-authored explanations are written separately from actual saved predictions"
run_manifest["perturbation_results_sha256"] = hashlib.sha256((RUNTIME / "perturbation_results.json").read_bytes()).hexdigest()
(RUNTIME / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
print({name: (ROOT / name).stat().st_size for name in required})

{'results.csv': 670, 'learning_curves.png': 196894, 'em_f1_analysis.png': 30682, 'answer_length_analysis.png': 59203}
